# 01 - Single Izhikevich Neuron

## Objetivo
Implementar y visualizar una neurona Izhikevich.


## Cómo leer esta notebook

Esta primera etapa responde dos preguntas diferentes:

1. **¿Cómo se comporta una neurona de Izhikevich aislada?** Se observa cómo el input modifica `v`, `u` y los spikes.
2. **¿Qué cambia al simular muchas neuronas independientes?** Todas reciben el mismo comando, pero el ruido y las pequeñas diferencias de parámetros evitan respuestas idénticas.

El recorrido conceptual es:

```text
tipo de input + amplitud
          ↓
 corriente I(t)
          ↓
 ecuaciones de Izhikevich
          ↓
 potencial v(t) y recuperación u(t)
          ↓
 si v ≥ 30 mV: registrar spike y aplicar reset
```

Todavía no hay conexiones sinápticas entre neuronas. Esta notebook valida los bloques elementales que usarán las siguientes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


NEURON_PRESETS = {
    "Regular Spiking": (0.02, 0.20, -65.0, 8.0),
    "Intrinsically Bursting": (0.02, 0.20, -55.0, 4.0),
    "Chattering": (0.02, 0.20, -50.0, 2.0),
    "Fast Spiking": (0.10, 0.20, -65.0, 2.0),
    "Low Threshold Spiking": (0.02, 0.25, -65.0, 2.0),
}


def generate_single_input(
    input_type="constant", t=None, amplitude=10.0, start=100.0,
    end=900.0, frequency=5.0, noise_level=2.0
):
    """Genera el input externo usado por la neurona individual."""
    I = np.zeros_like(t)
    active = (t >= start) & (t <= end)

    if input_type == "constant":
        I[active] = amplitude
    elif input_type == "pulse":
        I[(t >= start) & (t <= min(start + 100.0, end))] = amplitude
    elif input_type == "ramp":
        I[active] = amplitude * (t[active] - start) / max(end - start, 1)
    elif input_type == "sinusoidal":
        phase = 2 * np.pi * frequency * (t[active] - start) / 1000
        I[active] = amplitude * (0.5 + 0.5 * np.sin(phase))
    elif input_type == "noisy":
        I[active] = amplitude + noise_level * np.random.randn(np.sum(active))
    elif input_type == "motor_plan":
        transition = min(200.0, (end - start) / 2)
        rise_end, fall_start = start + transition, end - transition
        rise = (t >= start) & (t < rise_end)
        hold = (t >= rise_end) & (t < fall_start)
        fall = (t >= fall_start) & (t <= end)
        I[rise] = amplitude * (t[rise] - start) / transition
        I[hold] = amplitude
        I[fall] = amplitude * (end - t[fall]) / transition
    return I


def simulate_izhikevich(
    a=0.02, b=0.2, c=-65.0, d=8.0, input_type="constant",
    amplitude=10.0, total_time=1000, dt=0.5, input_start=100,
    input_end=900, frequency=5.0, noise_level=2.0
):
    """Simula una neurona de Izhikevich usando Euler."""
    t = np.arange(0, total_time, dt)
    v = np.zeros(t.size)
    u = np.zeros(t.size)
    spikes = np.zeros(t.size, dtype=bool)
    v[0] = -65.0
    u[0] = b * v[0]
    I = generate_single_input(input_type, t, amplitude, input_start, input_end, frequency, noise_level)

    for step in range(1, t.size):
        dv = 0.04 * v[step-1]**2 + 5 * v[step-1] + 140 - u[step-1] + I[step-1]
        du = a * (b * v[step-1] - u[step-1])
        v[step] = v[step-1] + dt * dv
        u[step] = u[step-1] + dt * du
        if v[step] >= 30:
            v[step-1] = 30
            v[step] = c
            u[step] += d
            spikes[step] = True
    return t, v, u, I, spikes


def plot_simulation(
    neuron_type="Regular Spiking", input_type="constant", amplitude=10.0,
    total_time=1000, dt=0.5, frequency=5.0, noise_level=2.0
):
    """Ejecuta y grafica una neurona usando el preset seleccionado."""
    a, b, c, d = NEURON_PRESETS[neuron_type]
    input_start = 0.10 * total_time
    input_end = 0.90 * total_time
    t, v, u, I, spikes = simulate_izhikevich(
        a=a, b=b, c=c, d=d, input_type=input_type, amplitude=amplitude,
        total_time=total_time, dt=dt, input_start=input_start,
        input_end=input_end, frequency=frequency, noise_level=noise_level
    )
    spike_times = t[spikes]
    firing_rate = len(spike_times) / (total_time / 1000)

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(t, v)
    axes[0].set_ylabel("v (mV)")
    axes[0].set_title(
        f"Izhikevich — {neuron_type} | Spikes: {len(spike_times)} | "
        f"Firing rate: {firing_rate:.2f} Hz"
    )
    axes[1].plot(t, I, color="tab:orange")
    axes[1].set_ylabel("Input I(t)")
    axes[2].eventplot(spike_times, lineoffsets=1, linelengths=0.8)
    axes[2].set(xlabel="Tiempo (ms)", ylabel="Spikes", yticks=[])
    for ax in axes:
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def gui_row(title, description, control):
    label = widgets.HTML(
        f"<div style='width:290px'><b>{title}</b><br>"
        f"<span style='font-size:12px'>{description}</span></div>"
    )
    return widgets.HBox([label, control])


# GUI 1: controles que modifican directamente la simulación.
single_neuron_type = widgets.Dropdown(options=list(NEURON_PRESETS), value="Regular Spiking")
single_input_type = widgets.Dropdown(
    options=["constant", "pulse", "ramp", "sinusoidal", "noisy", "motor_plan"],
    value="constant"
)
single_amplitude = widgets.FloatSlider(value=10.0, min=0.0, max=30.0, step=0.5)
single_total_time = widgets.IntSlider(value=1000, min=300, max=3000, step=100)
single_dt = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.1)
single_frequency = widgets.FloatSlider(value=5.0, min=0.5, max=30.0, step=0.5)
single_noise = widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5)

single_frequency_row = gui_row("Frecuencia", "Solo se usa con input sinusoidal.", single_frequency)
single_noise_row = gui_row("Ruido del input", "Solo se usa con input noisy.", single_noise)
single_output = widgets.Output()
single_button = widgets.Button(description="Simular neurona", button_style="success", icon="play")


def update_single_visibility(change=None):
    single_frequency_row.layout.display = "" if single_input_type.value == "sinusoidal" else "none"
    single_noise_row.layout.display = "" if single_input_type.value == "noisy" else "none"


def run_single_gui(_=None):
    with single_output:
        single_output.clear_output(wait=True)
        plot_simulation(
            neuron_type=single_neuron_type.value,
            input_type=single_input_type.value,
            amplitude=single_amplitude.value,
            total_time=single_total_time.value,
            dt=single_dt.value,
            frequency=single_frequency.value,
            noise_level=single_noise.value,
        )


single_input_type.observe(update_single_visibility, names="value")
single_button.on_click(run_single_gui)
update_single_visibility()

single_gui = widgets.VBox([
    widgets.HTML("<h3>GUI 1 — Neurona individual</h3>"),
    gui_row("Tipo neuronal", "Selecciona el preset que fija a, b, c y d.", single_neuron_type),
    gui_row("Tipo de input", "Selecciona la forma temporal del estímulo.", single_input_type),
    gui_row("Amplitud", "Intensidad del estímulo externo.", single_amplitude),
    gui_row("Duración", "Tiempo total de simulación en ms.", single_total_time),
    gui_row("dt", "Paso de integración de Euler en ms.", single_dt),
    single_frequency_row,
    single_noise_row,
    single_button,
])
display(single_gui, single_output)


### Qué observar en la neurona individual

- **Potencial `v(t)`**: la variable rápida que produce el spike. El pico dibujado en 30 mV es un marcador; después se aplica `v = c`.
- **Recuperación `u(t)`**: resume procesos lentos que frenan o adaptan el disparo. Después de un spike aumenta en `d`.
- **Input `I(t)`**: no es voltaje ni fuerza; es la corriente externa abstracta que impulsa el modelo.
- **Spikes totales**: cantidad de veces que se alcanzó el umbral.
- **Firing rate**: spikes por segundo durante toda la simulación. Si hay largos períodos de reposo, esta tasa será menor que la calculada solo durante el estímulo.

Los presets cambian `a`, `b`, `c` y `d`. En el proyecto se usan como patrones funcionales; no representan una calibración específica de una motoneurona real.

# 02 - Pull of Izhikevich Neurons

## Objetivo
Implementar y visualizar un grupo de neuronas Izhikevich

## Del caso individual al pool independiente

En un pool, cada fila de `V` y `spikes` corresponde a una neurona y cada columna a un instante temporal.

### ¿Qué significa la semilla aleatoria?

La simulación usa números aleatorios para generar:

- pequeñas variaciones de `a`, `b`, `c` y `d`;
- ruido individual en la corriente.

La **semilla** (`random_seed`) es el punto de inicio de ese generador pseudoaleatorio. No es una propiedad biológica ni un parámetro neuronal.

```text
misma semilla + mismos parámetros → exactamente la misma realización
otra semilla                    → otro ruido y otra heterogeneidad
```

Fijarla permite comparar dos condiciones sin que el cambio observado se deba a haber sorteado otro ruido. En esta notebook se fija internamente en 42 y se elimina de la GUI porque no es necesario modificarla en esta etapa. Para evaluar robustez se repetiría el experimento con muchas semillas y se resumirían la media y la dispersión.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import ipywidgets as widgets


In [16]:
# ============================================================
# 1. GENERADOR DE INPUTS
# ============================================================

def generate_input(
    input_type="motor_plan",
    t=None,
    amplitude=10.0,
    start=100.0,
    end=900.0,
    frequency=5.0
):
    """
    Genera un input común I(t) para todas las neuronas.

    Tipos disponibles:
    - constant: corriente constante
    - pulse: pulso breve
    - ramp: rampa creciente
    - sinusoidal: input oscilatorio
    - motor_plan: subida, mantenimiento y bajada
    """

    I = np.zeros_like(t)
    active = (t >= start) & (t <= end)

    if input_type == "constant":
        I[active] = amplitude

    elif input_type == "pulse":
        pulse_end = start + 100
        active_pulse = (t >= start) & (t <= pulse_end)
        I[active_pulse] = amplitude

    elif input_type == "ramp":
        duration = max(end - start, 1)
        I[active] = amplitude * (t[active] - start) / duration

    elif input_type == "sinusoidal":
        I[active] = amplitude * (
            0.5 + 0.5 * np.sin(2 * np.pi * frequency * (t[active] - start) / 1000)
        )

    elif input_type == "motor_plan":
        # Plan motor artificial:
        # reposo -> subida -> contracción sostenida -> relajación

        rise_start = start
        rise_end = start + 200

        hold_start = rise_end
        hold_end = end - 200

        fall_start = hold_end
        fall_end = end

        rise = (t >= rise_start) & (t < rise_end)
        hold = (t >= hold_start) & (t < hold_end)
        fall = (t >= fall_start) & (t <= fall_end)

        if rise_end > rise_start:
            I[rise] = amplitude * (t[rise] - rise_start) / (rise_end - rise_start)

        I[hold] = amplitude

        if fall_end > fall_start:
            I[fall] = amplitude * (1 - (t[fall] - fall_start) / (fall_end - fall_start))

    return I

In [ ]:
# ============================================================
# 2. SIMULACIÓN DEL POOL DE NEURONAS IZHIKEVICH
# ============================================================

def simulate_neuron_pool(
    n_neurons=30,
    neuron_type="Regular Spiking",
    total_time=1000,
    dt=0.5,
    input_type="motor_plan",
    amplitude=10.0,
    input_start=100,
    input_end=900,
    frequency=5.0,
    parameter_noise=0.05,
    input_noise=1.0,
    random_seed=42
):
    """Simula un pool independiente basado en el preset seleccionado."""
    rng = np.random.default_rng(random_seed)
    t = np.arange(0, total_time, dt)
    n_steps = len(t)
    I_common = generate_input(
        input_type=input_type, t=t, amplitude=amplitude,
        start=input_start, end=input_end, frequency=frequency
    )

    base_a, base_b, base_c, base_d = NEURON_PRESETS[neuron_type]
    a = np.clip(base_a * (1 + parameter_noise * rng.normal(size=n_neurons)), 0.001, 0.2)
    b = np.clip(base_b * (1 + parameter_noise * rng.normal(size=n_neurons)), 0.01, 0.4)
    c = np.clip(base_c + abs(base_c) * parameter_noise * rng.normal(size=n_neurons), -90, -40)
    d = np.clip(base_d * (1 + parameter_noise * rng.normal(size=n_neurons)), 0.1, 20)
    params = {"a": a, "b": b, "c": c, "d": d}

    V = np.zeros((n_neurons, n_steps))
    U = np.zeros((n_neurons, n_steps))
    spikes = np.zeros((n_neurons, n_steps), dtype=bool)
    V[:, 0] = -65.0
    U[:, 0] = b * V[:, 0]
    I_all = np.tile(I_common, (n_neurons, 1))
    I_all += input_noise * rng.normal(size=(n_neurons, n_steps))

    for step in range(1, n_steps):
        dv = 0.04 * V[:, step-1]**2 + 5 * V[:, step-1] + 140 - U[:, step-1] + I_all[:, step-1]
        du = a * (b * V[:, step-1] - U[:, step-1])
        V[:, step] = V[:, step-1] + dt * dv
        U[:, step] = U[:, step-1] + dt * du
        fired = V[:, step] >= 30
        V[fired, step-1] = 30
        V[fired, step] = c[fired]
        U[fired, step] += d[fired]
        spikes[fired, step] = True

    return t, V, U, spikes, I_common, I_all, params


In [18]:
# ============================================================
# 3. ACTIVIDAD POBLACIONAL
# ============================================================

def compute_population_activity(spikes, dt=0.5, bin_size_ms=20):
    """
    Cuenta cuántos spikes ocurren en ventanas de tiempo.
    """

    n_neurons, n_steps = spikes.shape

    steps_per_bin = max(int(bin_size_ms / dt), 1)
    n_bins = n_steps // steps_per_bin

    activity = np.zeros(n_bins)
    time_bins = np.zeros(n_bins)

    for i in range(n_bins):
        start = i * steps_per_bin
        end = start + steps_per_bin

        activity[i] = spikes[:, start:end].sum()
        time_bins[i] = start * dt

    return time_bins, activity

### Cómo leer las visualizaciones poblacionales

- **Raster**: una fila por neurona; cada marca vertical es un spike. Permite ver sincronía, silencios y distribución temporal.
- **Mapa de calor**: muestra el potencial de todas las neuronas, no solo los spikes.
- **Actividad poblacional**: suma los spikes dentro de ventanas de tiempo (`bin_size_ms`). Un bin grande suaviza la señal; uno pequeño conserva más detalle y fluctúa más.
- **Neuronas activas**: neuronas que dispararon al menos una vez. No informa cuántas veces disparó cada una.

El pool de esta sección sigue siendo independiente: similitudes entre filas provienen del input común, no de conexiones entre neuronas.

In [19]:
# ============================================================
# 4. DIAGRAMA DE ARQUITECTURA: INPUT -> POOL
# ============================================================

def plot_input_to_pool_grid(n_neurons=30, show_labels=True):
    """
    Visualiza la arquitectura actual:

    Input común -> pool de N neuronas.

    Este gráfico es solo una referencia visual.
    En esta etapa las neuronas NO están conectadas entre sí.
    """

    G = nx.DiGraph()

    input_node = "Input"
    G.add_node(input_node)

    neuron_nodes = [f"N{i+1}" for i in range(n_neurons)]

    for neuron in neuron_nodes:
        G.add_node(neuron)
        G.add_edge(input_node, neuron)

    # Posiciones
    pos = {}
    pos[input_node] = (0, 2)

    n_cols = int(np.ceil(np.sqrt(n_neurons)))
    n_rows = int(np.ceil(n_neurons / n_cols))

    spacing_x = 1.0
    spacing_y = 0.8

    for idx, neuron in enumerate(neuron_nodes):
        row = idx // n_cols
        col = idx % n_cols

        x = (col - n_cols / 2) * spacing_x
        y = -row * spacing_y

        pos[neuron] = (x, y)

    plt.figure(figsize=(12, 8))

    # Nodo input
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=[input_node],
        node_size=1800,
        node_color="lightgray",
        edgecolors="black"
    )

    # Nodos neuronales
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=neuron_nodes,
        node_size=500 if n_neurons <= 50 else 250,
        node_color="white",
        edgecolors="black"
    )

    # Flechas
    nx.draw_networkx_edges(
        G,
        pos,
        arrows=True,
        arrowstyle="->",
        arrowsize=10,
        width=0.6,
        alpha=0.4
    )

    # Labels
    if show_labels:
        labels = {node: node for node in G.nodes()}
        font_size = 8 if n_neurons <= 50 else 5
    else:
        labels = {input_node: "Input"}
        font_size = 12

    nx.draw_networkx_labels(
        G,
        pos,
        labels=labels,
        font_size=font_size
    )

    plt.title(f"Arquitectura actual: input común conectado a {n_neurons} neuronas")
    plt.axis("off")
    plt.show()

In [20]:
# ============================================================
# 5. VISUALIZACIÓN DE RESULTADOS
# ============================================================

def plot_neuron_pool_results(
    t,
    V,
    spikes,
    I_common,
    dt=0.5,
    n_voltage_traces=5,
    bin_size_ms=20
):
    """
    Grafica:
    1. Input común.
    2. Potencial de membrana de algunas neuronas.
    3. Raster plot.
    4. Mapa de calor del voltaje.
    5. Actividad poblacional.
    """

    n_neurons, n_steps = V.shape

    time_bins, activity = compute_population_activity(
        spikes=spikes,
        dt=dt,
        bin_size_ms=bin_size_ms
    )

    fig, axes = plt.subplots(5, 1, figsize=(15, 18), sharex=False)

    # --------------------------------------------------------
    # 1. Input común
    # --------------------------------------------------------
    axes[0].plot(t, I_common)
    axes[0].set_title("Input común recibido por el pool neuronal")
    axes[0].set_ylabel("I(t)")
    axes[0].set_xlabel("Tiempo (ms)")
    axes[0].grid(True)

    # --------------------------------------------------------
    # 2. Voltaje de algunas neuronas
    # --------------------------------------------------------
    selected_neurons = np.linspace(
        0,
        n_neurons - 1,
        min(n_voltage_traces, n_neurons),
        dtype=int
    )

    for neuron_idx in selected_neurons:
        axes[1].plot(t, V[neuron_idx], label=f"Neuron {neuron_idx + 1}")

    axes[1].set_title("Potencial de membrana de algunas neuronas")
    axes[1].set_ylabel("v(t)")
    axes[1].set_xlabel("Tiempo (ms)")
    axes[1].legend(loc="upper right")
    axes[1].grid(True)

    # --------------------------------------------------------
    # 3. Raster plot
    # --------------------------------------------------------
    for neuron_idx in range(n_neurons):
        spike_times = t[spikes[neuron_idx] == 1]

        axes[2].vlines(
            spike_times,
            neuron_idx + 0.5,
            neuron_idx + 1.5,
            linewidth=0.8
        )

    axes[2].set_title("Raster plot: spikes por neurona")
    axes[2].set_ylabel("Neurona")
    axes[2].set_xlabel("Tiempo (ms)")
    axes[2].set_ylim(0.5, n_neurons + 0.5)
    axes[2].grid(True)

    # --------------------------------------------------------
    # 4. Mapa de calor del potencial de membrana
    # --------------------------------------------------------
    im = axes[3].imshow(
        V,
        aspect="auto",
        origin="lower",
        extent=[t[0], t[-1], 1, n_neurons],
        interpolation="nearest"
    )

    axes[3].set_title("Mapa de calor del potencial de membrana")
    axes[3].set_ylabel("Neurona")
    axes[3].set_xlabel("Tiempo (ms)")

    cbar = fig.colorbar(im, ax=axes[3])
    cbar.set_label("v(t)")

    # --------------------------------------------------------
    # 5. Actividad poblacional
    # --------------------------------------------------------
    axes[4].bar(
        time_bins,
        activity,
        width=bin_size_ms,
        align="edge"
    )

    axes[4].set_title(f"Actividad poblacional: spikes cada {bin_size_ms} ms")
    axes[4].set_ylabel("Cantidad de spikes")
    axes[4].set_xlabel("Tiempo (ms)")
    axes[4].grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 6. FUNCIÓN PRINCIPAL DE LA GUI
# ============================================================

def run_pool_gui(
    n_neurons=30,
    neuron_type="Regular Spiking",
    input_type="motor_plan",
    amplitude=10.0,
    total_time=1000,
    dt=0.5,
    frequency=5.0,
    parameter_noise=0.05,
    input_noise=1.0,
    show_network_graph=True,
):
    """Ejecuta el pool con timing, seed y bins fijados internamente."""
    input_start = 0.10 * total_time
    input_end = 0.90 * total_time
    random_seed = 42  # fija para que repetir la GUI produzca el mismo caso
    bin_size_ms = 20

    if show_network_graph:
        plot_input_to_pool_grid(n_neurons=n_neurons, show_labels=n_neurons <= 30)

    t, V, U, spikes, I_common, I_all, params = simulate_neuron_pool(
        n_neurons=n_neurons,
        neuron_type=neuron_type,
        total_time=total_time,
        dt=dt,
        input_type=input_type,
        amplitude=amplitude,
        input_start=input_start,
        input_end=input_end,
        frequency=frequency,
        parameter_noise=parameter_noise,
        input_noise=input_noise,
        random_seed=random_seed,
    )

    total_spikes = int(spikes.sum())
    firing_rate_mean = total_spikes / n_neurons / (total_time / 1000)
    print("Resumen de simulación")
    print("---------------------")
    print(f"Tipo neuronal: {neuron_type}")
    print(f"Número de neuronas: {n_neurons}")
    print(f"Tipo de input: {input_type}")
    print(f"Spikes totales: {total_spikes}")
    print(f"Firing rate medio: {firing_rate_mean:.2f} Hz")
    print("Pool independiente: no hay conexiones sinápticas entre neuronas.")

    plot_neuron_pool_results(
        t=t, V=V, spikes=spikes, I_common=I_common,
        dt=dt, bin_size_ms=bin_size_ms
    )
    return t, V, U, spikes, I_common, I_all, params


In [ ]:
# ============================================================
# 7. GUI INTERACTIVA CON EJECUCIÓN MANUAL
# ============================================================

pool_neuron_type = widgets.Dropdown(options=list(NEURON_PRESETS), value="Regular Spiking")
pool_n_neurons = widgets.IntSlider(value=30, min=5, max=50, step=5)
pool_input_type = widgets.Dropdown(
    options=["constant", "pulse", "ramp", "sinusoidal", "motor_plan"],
    value="motor_plan"
)
pool_amplitude = widgets.FloatSlider(value=10.0, min=0.0, max=30.0, step=0.5)
pool_total_time = widgets.IntSlider(value=1000, min=500, max=3000, step=100)
pool_dt = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.1)
pool_frequency = widgets.FloatSlider(value=5.0, min=0.5, max=30.0, step=0.5)
pool_parameter_noise = widgets.FloatSlider(value=0.05, min=0.0, max=0.20, step=0.01)
pool_input_noise = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.25)
pool_show_graph = widgets.Checkbox(value=True, description="Mostrar arquitectura")

pool_frequency_row = gui_row("Frecuencia", "Solo se usa con input sinusoidal.", pool_frequency)
pool_output = widgets.Output()
pool_button = widgets.Button(description="Simular pool", button_style="success", icon="play")


def update_pool_visibility(change=None):
    pool_frequency_row.layout.display = "" if pool_input_type.value == "sinusoidal" else "none"


def run_pool_button(_=None):
    with pool_output:
        pool_output.clear_output(wait=True)
        run_pool_gui(
            n_neurons=pool_n_neurons.value,
            neuron_type=pool_neuron_type.value,
            input_type=pool_input_type.value,
            amplitude=pool_amplitude.value,
            total_time=pool_total_time.value,
            dt=pool_dt.value,
            frequency=pool_frequency.value,
            parameter_noise=pool_parameter_noise.value,
            input_noise=pool_input_noise.value,
            show_network_graph=pool_show_graph.value,
        )


pool_input_type.observe(update_pool_visibility, names="value")
pool_button.on_click(run_pool_button)
update_pool_visibility()

pool_gui = widgets.VBox([
    widgets.HTML("<h3>GUI 2 — Pool neuronal independiente</h3>"),
    gui_row("Tipo neuronal", "Preset común; luego se agrega heterogeneidad individual.", pool_neuron_type),
    gui_row("Número de neuronas", "Tamaño del pool independiente.", pool_n_neurons),
    gui_row("Tipo de input", "Forma del comando común recibido por el pool.", pool_input_type),
    gui_row("Amplitud", "Intensidad del comando externo.", pool_amplitude),
    gui_row("Duración", "Tiempo total de simulación en ms.", pool_total_time),
    gui_row("dt", "Paso de integración de Euler en ms.", pool_dt),
    pool_frequency_row,
    gui_row("Heterogeneidad", "Variación relativa de a, b, c y d dentro del pool.", pool_parameter_noise),
    gui_row("Ruido individual", "Ruido de corriente diferente para cada neurona.", pool_input_noise),
    pool_show_graph,
    pool_button,
])
display(pool_gui, pool_output)
